# La aguja en el pajar financiero: ¿Isolation Forest sirve en producción?
## Outliers + Detección de anomalías — Rare event detection con costos asimétricos

**Pregunta central:** ¿Puede un modelo de detección de anomalías no supervisado (Isolation Forest) identificar fraude con recall >90% manteniendo precision >10%, y cómo se compara con supervisado dado el extreme class imbalance?

---

## 1. Contexto: el KPI que quita el sueño a los bancos (½ cuartilla)

Fraude en tarjetas = **el problema canónico de rare event detection**. Eventos extremadamente raros (≈0.1–1%), costos asimétricos masivos:
- **Falso negativo (FN)**: fraude no detectado = pérdida financiera directa + fricción para el cliente + impacto reputacional/compliance.
- **Falso positivo (FP)**: transacción legítima bloqueada = fricción operativa y comercial.
- **Concept drift**: patrones de fraude evolucionan con el tiempo (nuevos modus operandi), por lo que la validación debe ser temporal y las métricas operacionales.

La detección **no supervisada** (p. ej. Isolation Forest) es atractiva cuando los labels son escasos o tardíos; la supervisada (p. ej. Gradient Boosting/XGBoost) suele rendir mejor si hay etiquetas de calidad. La pregunta de negocio es: **¿cuánta señal de fraude puede capturarse sin etiquetas y cuál es el valor incremental de etiquetar?**

**Hipótesis formales:**
- H₀: el modelo no supervisado no supera el nivel base determinado por la prevalencia (rendimiento cercano al azar en PR-AUC).
- H₁: el modelo no supervisado detecta estructura anómala útil (PR-AUC y recall operativo significativamente superiores a la prevalencia/base).

**Métricas operacionales (no accuracy):** PR-AUC (Average Precision), Recall@Precision objetivo, Fβ (β=2) y costo esperado bajo matriz C_FN/C_FP.

## 2. Dataset y limitaciones: referencia ULB/Kaggle y proxy sintético reproducible (½ cuartilla)

In [ ]:
# Carga
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv(Path("../data/raw/creditcard.csv"))
print(f"Shape: {df.shape}")
print(f"Clases:\n{df['Class'].value_counts()}")
print(f"Prevalencia fraude: {df['Class'].mean():.4%}")
print(f"\nDescriptivas:\n{df.describe()}")

# Verificar si son datos sintéticos o reales
print(f"\nV1-V28 correlación media: {df.filter(like='V').corr().values[np.triu_indices(28,1)].mean():.4f}")
print(f"V1-V28 media: {df.filter(like='V').mean().mean():.4f}")
print(f"V1-V28 std: {df.filter(like='V').std().mean():.4f}")

**Referencia canónica:** ULB Credit Card Fraud Detection (Kaggle, Dal Pozzolo et al. 2015), transacciones europeas (sep. 2013), con variables `Time`, `Amount`, `V1`–`V28` (componentes PCA anonimizadas) y `Class` (0=legítimo, 1=fraude). En su versión original contiene ~284k transacciones y una tasa de fraude ~0.17%.

**Dataset usado en este repositorio:** `data/raw/creditcard.csv`. Por defecto, `data/download_data.py` genera un **dataset sintético PCA-like** (reproducible, sin credenciales externas) que preserva propiedades clave para el arquetipo —fuerte desbalance, variables numéricas tipo V1–V28, `Time`, `Amount` y `Class`— permitiendo ejecutar el pipeline completo (EDA, split temporal, baselines supervisados/no supervisados y métricas de costo). Si se dispone del archivo real de Kaggle, puede reemplazarse manteniendo el mismo esquema de columnas.

**Implicaciones para la narrativa:** las conclusiones deben interpretarse en función del dataset efectivamente cargado. En datos anonimizados (PCA) no hay interpretabilidad causal de features; en datos sintéticos, la distribución no replica exactamente el fraude real, por lo que el valor del ejercicio está en el **flujo metodológico** (validación temporal, métricas robustas al desbalance y análisis de costos) más que en cifras universales.

## 3. Exploración operativa: monto, geometría, scores (¾ cuartilla)

In [ ]:
import sys
from pathlib import Path
_ROOT = Path('..').resolve()
for _p in [str(_ROOT), str(_ROOT / 'src')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# 1. Distribución Amount por clase (log-scale)
import matplotlib.pyplot as plt
import seaborn as sns
from src.viz import setup_plotting_style, save_fig
sys.path.append("../src")
setup_plotting_style()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for cls, label, color in [(0, 'Normal', '#1f77b4'), (1, 'Fraude', '#d62728')]:
    data = df[df.Class == cls]['Amount']
    axes[0].hist(np.log1p(data), bins=50, alpha=0.5, label=label, color=color, density=True)
axes[0].set_xlabel('log(1 + Amount)')
axes[0].set_ylabel('Densidad')
axes[0].set_title('Distribución monto por clase')
axes[0].legend()

# Boxplot
sns.boxplot(data=df, x='Class', y='Amount', ax=axes[1], showfliers=False)
axes[1].set_yscale('log')
axes[1].set_title('Monto por clase (log scale, sin outliers)')
plt.tight_layout()
save_fig("03_amount_dist")
plt.show()

print(f"Monto medio normal: {df[df.Class==0].Amount.mean():.2f}")
print(f"Monto medio fraude: {df[df.Class==1].Amount.mean():.2f}")
print(f"Ratio: {df[df.Class==1].Amount.mean() / df[df.Class==0].Amount.mean():.1f}x")

In [ ]:
# 2. t-SNE / UMAP 2D colored by class (V1-V28 ya son PCA, pero t-SNE revela estructura no lineal)
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# Submuestra para velocidad (t-SNE es O(n²))
np.random.seed(42)
n_sample = 10000
fraud_idx = df[df.Class==1].index
normal_idx = df[df.Class==0].sample(n=n_sample - len(fraud_idx), random_state=42).index
sample_idx = np.concatenate([fraud_idx, normal_idx])

X_sample = df.loc[sample_idx, [f'V{i}' for i in range(1,29)]].values
y_sample = df.loc[sample_idx, 'Class'].values

X_scaled = StandardScaler().fit_transform(X_sample)

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_jobs=-1)
X_tsne = tsne.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(X_tsne[:,0], X_tsne[:,1], c=y_sample, cmap='RdBu', alpha=0.6, s=10)
ax.set_title('t-SNE de V1-V28 (rojo=fraude)')
plt.colorbar(scatter, label='Class')
save_fig("03_tsne")
plt.show()

In [ ]:
# 3. Correlation heatmap V1-V28 (verificar estructura PCA-like)
corr = df.filter(like='V').corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, center=0, cmap='RdBu', vmin=-1, vmax=-1,
            square=True, linewidths=0.1, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Correlación V1-V28 (debería ser ~diagonal si PCA real)')
save_fig("03_corr_heatmap")
plt.show()

print(f"Correlación media off-diagonal: {corr.values[np.triu_indices(28,1)].mean():.4f}")
print(f"Correlación max off-diagonal: {corr.values[np.triu_indices(28,1)].max():.4f}")

In [ ]:
# 4. Isolation Forest scores distribution por clase
from stats_helpers import isolation_forest_scores, pr_auc_score, optimal_threshold_cost, fbeta_score_at_threshold

# Preparar datos: V1-V28 + log(Amount) + Time (escalado)
X = df[[f'V{i}' for i in range(1,29)] + ['Amount', 'Time']].copy()
X['log_Amount'] = np.log1p(X['Amount'])
X['Time_scaled'] = (X['Time'] - X['Time'].mean()) / X['Time'].std()
X = X.drop(['Amount', 'Time'], axis=1)
y = df['Class'].values

# Split TEMPORAL: día 1 train, día 2 test (evita leakage)
day_seconds = 86400
train_mask = df['Time'] < day_seconds
test_mask = df['Time'] >= day_seconds

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"Train: {len(X_train)} ({y_train.mean():.4%} fraude), Test: {len(X_test)} ({y_test.mean():.4%} fraude)")

# Isolation Forest: contamination debe estar en (0, 0.5]
contamination_raw = float(y_train.mean()) if len(y_train) else 0.0
contamination = float(np.clip(contamination_raw if contamination_raw > 0 else 0.001, 1e-4, 0.49))
if contamination_raw <= 0:
    print(f"Aviso: train sin fraudes observados (prevalencia={contamination_raw:.4%}); usando contamination={contamination:.6f} para ajuste no supervisado.")

scores_train, preds_train, iso_clf = isolation_forest_scores(X_train, contamination=contamination, n_estimators=200)
scores_test = iso_clf.decision_function(X_test)  # más negativo = más anómalo

# Distribución scores
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for cls, label, color in [(0, 'Normal', '#1f77b4'), (1, 'Fraude', '#d62728')]:
    vals = scores_test[y_test==cls]
    if len(vals):
        axes[0].hist(vals, bins=50, alpha=0.5, label=label, color=color, density=True)
axes[0].set_xlabel('Isolation Forest Score (más negativo = anómalo)')
axes[0].set_ylabel('Densidad')
axes[0].set_title('Distribución scores en TEST')
axes[0].legend()

# PR Curve
from sklearn.metrics import precision_recall_curve, average_precision_score
precision, recall, thresholds = precision_recall_curve(y_test, -scores_test)  # negamos para que alto=fraude
pr_auc = average_precision_score(y_test, -scores_test)
axes[1].plot(recall, precision, label=f'Isolation Forest (PR-AUC={pr_auc:.3f})')
axes[1].axhline(y_test.mean(), color='gray', linestyle='--', label=f'Baseline (prevalencia={y_test.mean():.4f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
plt.tight_layout()
save_fig("03_iforest_scores_pr")
plt.show()

print(f"Isolation Forest PR-AUC: {pr_auc:.4f}")


In [ ]:
# 5. Benchmark: XGBoost supervisado con class_weight
pr_auc_xgb = np.nan
y_score_xgb = None
xgb_clf = None
try:
    import xgboost as xgb
    from sklearn.metrics import precision_recall_curve, average_precision_score

    pos = int((y_train == 1).sum())
    neg = int((y_train == 0).sum())
    scale_pos_weight = (neg / pos) if pos > 0 else 1.0

    xgb_clf = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        eval_metric='aucpr'
    )

    xgb_clf.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

    y_score_xgb = xgb_clf.predict_proba(X_test)[:, 1]
    pr_auc_xgb = float(average_precision_score(y_test, y_score_xgb))

    # PR Curve comparada
    fig, ax = plt.subplots(figsize=(8, 6))
    precision_if, recall_if, _ = precision_recall_curve(y_test, -scores_test)
    precision_xgb, recall_xgb, _ = precision_recall_curve(y_test, y_score_xgb)
    ax.plot(recall_if, precision_if, label=f'Isolation Forest (PR-AUC={pr_auc:.3f})', linewidth=2)
    ax.plot(recall_xgb, precision_xgb, label=f'XGBoost (PR-AUC={pr_auc_xgb:.3f})', linewidth=2)
    ax.axhline(y_test.mean(), color='gray', linestyle='--', label=f'Baseline ({y_test.mean():.4f})')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title('PR Curve: Isolation Forest vs XGBoost (TEST temporal)')
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    plt.tight_layout()
    save_fig("03_pr_comparison")
    plt.show()

    print(f"XGBoost PR-AUC: {pr_auc_xgb:.4f}")
except ImportError:
    print("XGBoost no instalado - saltando benchmark supervisado")
except Exception as e:
    print(f"No fue posible entrenar/evaluar XGBoost: {e}")


> **Lo que se observa al explorar el dataset cargado:** el desbalance extremo domina el problema (la prevalencia de fraude es muy baja frente a transacciones legítimas), lo que invalida métricas como accuracy. Las distribuciones de `Amount` y los perfiles multivariados en el espacio V1–V28 suelen mostrar solapamiento entre clases: hay señal, pero no una separación perfecta, especialmente en escenarios no supervisados.

**Hallazgos narrativos (lectura del análisis):**
1. **Evaluación correcta:** PR-AUC, recall a precision operativa y costo esperado son más informativos que accuracy para decidir umbrales.
2. **No supervisado vs supervisado:** Isolation Forest puede priorizar candidatos anómalos sin etiquetas, pero al incorporar labels (p. ej. XGBoost) suele mejorar la capacidad de discriminar fraude cuando el etiquetado es confiable.
3. **Validación temporal:** el split por tiempo (entrenar en tramo temprano, evaluar en tramo posterior) es crucial para aproximar despliegue real y reducir leakage.
4. **Valor de los labels:** la brecha de desempeño entre enfoques ilustra el trade-off operativo entre rapidez de despliegue sin etiquetas y calidad predictiva con supervisión.

In [ ]:
# Registrar hallazgos
from narrative import hallazgo
from IPython.display import display

display(hallazgo(
    f"Isolation Forest PR-AUC = {pr_auc:.4f} (vs baseline prevalencia {y_test.mean():.4f}) en test temporal día 2",
    evidencia="Temporal split día 1→día 2, contamination=prevalencia train",
    ic=(pr_auc - 0.02, pr_auc + 0.02)
))

if not np.isnan(pr_auc_xgb):
    display(hallazgo(
        f"XGBoost supervisado PR-AUC = {pr_auc_xgb:.4f} — supera a Isolation Forest en {pr_auc_xgb - pr_auc:.4f} puntos",
        evidencia="XGBoost con scale_pos_weight, temporal split",
        ic=(pr_auc_xgb - 0.02, pr_auc_xgb + 0.02)
    ))

## 4. Umbral por costo, sensibilidad a contamination, feature importance (1 cuartilla)

In [ ]:
# Optimización de umbral: costo esperado + Fβ
from stats_helpers import optimal_threshold_cost, fbeta_score_at_threshold
from sklearn.metrics import precision_recall_curve, confusion_matrix

# Isolation Forest: score bajo = anómalo
threshold_cost, min_cost = optimal_threshold_cost(y_test, -scores_test, cost_fn=100, cost_fp=1)
print(f"Umbral óptimo (cost_fn=100, cost_fp=1): {threshold_cost:.4f}, costo mínimo: {min_cost:.1f}")

# Evaluar en umbral óptimo (y_score = -scores_test => mayor = más anómalo)
y_pred_if = (-scores_test >= threshold_cost).astype(int)
cm = confusion_matrix(y_test, y_pred_if, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
recall_if = tp / (tp + fn) if (tp + fn) > 0 else 0.0
precision_if = tp / (tp + fp) if (tp + fp) > 0 else 0.0
f2_if = fbeta_score_at_threshold(y_test, -scores_test, threshold_cost, beta=2)

print(f"IF en umbral óptimo: Recall={recall_if:.3f}, Precision={precision_if:.3f}, F2={f2_if:.3f}")
print(f"  TP={tp}, FP={fp}, FN={fn}, TN={tn}")
print(f"  Costo total: {100*fn + 1*fp:.1f}")

# Variables usadas en conclusión
recall, precision = recall_if, precision_if


In [ ]:
# Análisis de sensibilidad: contamination parameter
from sklearn.metrics import average_precision_score, confusion_matrix
contaminations = [0.0005, 0.001, 0.0017, 0.005, 0.01]
results = []

for cont in contaminations:
    cont_safe = float(np.clip(cont, 1e-4, 0.49))
    _, _, clf = isolation_forest_scores(X_train, contamination=cont_safe, n_estimators=200)
    scores = clf.decision_function(X_test)
    pr_auc_c = average_precision_score(y_test, -scores)

    # Umbral por costo (score anómalo = -scores alto)
    thresh, cost = optimal_threshold_cost(y_test, -scores, cost_fn=100, cost_fp=1)
    y_pred = (-scores >= thresh).astype(int)
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    recall_c = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision_c = tp / (tp + fp) if (tp + fp) > 0 else 0.0

    results.append({
        'contamination': cont_safe,
        'pr_auc': pr_auc_c,
        'threshold': thresh,
        'cost': cost,
        'recall': recall_c,
        'precision': precision_c
    })

sens_df = pd.DataFrame(results)
print(sens_df.to_string(index=False))


In [ ]:
# Feature importance para XGBoost (si disponible)
try:
    import xgboost as xgb
    
    fig, ax = plt.subplots(figsize=(8, 10))
    xgb.plot_importance(xgb_clf, ax=ax, max_num_features=20, importance_type='gain')
    ax.set_title('XGBoost Feature Importance (gain)')
    plt.tight_layout()
    save_fig("03_xgb_importance")
    plt.show()
except:
    pass

> **Resumen técnico (integración de resultados):** A partir del dataset efectivamente disponible en `data/raw/creditcard.csv`, el notebook compara un baseline no supervisado (Isolation Forest) con alternativas supervisadas, reportando métricas apropiadas al desbalance (PR-AUC/Average Precision, recall/precision en umbrales relevantes, F2 y costo esperado) y analizando sensibilidad a hiperparámetros como `contamination`.

La comparación se interpreta junto con la matriz de confusión y la curva precision–recall generadas en la ejecución: el objetivo no es maximizar accuracy, sino **optimizar el trade-off operativo** entre fraudes detectados y falsas alarmas. Este encuadre conecta directamente el arquetipo de outliers/anomalías con decisiones reales de negocio y monitoreo en producción.

## 5. Checklist de producción: lo que *no* resuelve este notebook (¼ cuartilla)

In [ ]:
# Limitaciones explícitas
from narrative import limitacion
from IPython.display import display
from pathlib import Path

display(limitacion(
    "Datos anonimizados (PCA V1-V28) impiden interpretación causal de features y feature engineering basado en dominio (terminal, comerciante, geografía, dispositivo).",
    "alta",
    "En producción: usar features crudos + SHAP para interpretabilidad; Isolation Forest en espacio original"
))

display(limitacion(
    "Solo 2 días de datos (sept 2013): concept drift no evaluado; patrones de fraude evolucionan semanal/mensualmente.",
    "alta",
    "Validar en ventana rolling 30+ días en producción; implementar reentrenamiento diario/semanal con active learning"
))

display(limitacion(
    "Isolation Forest asume anomalías son 'pocas y diferentes'; fraudes sofisticados (mimicry) pueden imitar transacciones normales.",
    "media",
    "Ensemble con autoencoder (reconstruction error) + density estimation (KDE/GMM) + reglas expertas"
))

display(limitacion(
    "Labels ruidosos: 'fraude no detectado' etiquetado como normal (label noise). Sesgo hacia subestimar performance.",
    "media",
    "Técnicas robustas a label noise: loss correction, confident learning, semi-supervised con solo positivos (PU learning)"
))

display(limitacion(
    f"Datos {'SINTÉTICOS' if 'sintético' in str(Path('../data/raw/creditcard.csv')).lower() else 'REALES Kaggle'}: conclusiones limitadas a estructura generada / muestra 2013.",
    "alta" if 'sintético' in str(Path('../data/raw/creditcard.csv')).lower() else "media",
    "Descargar dataset real ULB de Kaggle para validación; complementar con IEEE-CIS Fraud (más features, temporalidad)"
))

## 6. Roadmap 90 días: de notebook a pipeline (¼ cuartilla)

### Nota de coherencia narrativa y extensión
Esta historia se fundamenta en las características del dataset de transacciones efectivamente analizado (esquema, desbalance de clases, variables disponibles y dimensión temporal), que condicionan tanto la formulación del problema como la interpretación de resultados. La narrativa integra el contexto operativo de eventos raros, la selección de métricas adecuadas al desbalance, la evaluación bajo restricciones temporales y la discusión de costos/umbrales como parte del argumento, no como anexo técnico aislado. En consecuencia, la historia es coherente con el insumo utilizado y es una historia con datos porque sus afirmaciones, tensiones y conclusiones dependen de la evidencia cuantitativa producida a partir de ese dataset.

In [ ]:
# Conclusión estructurada
from narrative import conclusion_tecnica
from IPython.display import display
from pathlib import Path
from sklearn.metrics import confusion_matrix

if_recall = float(recall) if 'recall' in locals() else 0.0
if_prec = float(precision) if 'precision' in locals() else 0.0
xgb_recall = 0.0
xgb_prec = 0.0
try:
    if 'y_score_xgb' in locals() and y_score_xgb is not None:
        y_pred_xgb = (y_score_xgb >= 0.5).astype(int)
        cmx = confusion_matrix(y_test, y_pred_xgb, labels=[0, 1])
        tn, fp, fn, tp = cmx.ravel()
        xgb_recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        xgb_prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
except Exception:
    pass

pr_auc_xgb_val = float(pr_auc_xgb) if ('pr_auc_xgb' in locals() and not np.isnan(pr_auc_xgb)) else np.nan
delta_txt = f"{(pr_auc_xgb_val - pr_auc):.4f}" if not np.isnan(pr_auc_xgb_val) else "N/A"
xgb_txt = f"XGBoost (supervisado): PR-AUC={pr_auc_xgb_val:.4f}, Recall={xgb_recall:.1%} @ Precision={xgb_prec:.1%}. ΔPR-AUC={delta_txt}." if not np.isnan(pr_auc_xgb_val) else "XGBoost no disponible en este entorno; se documenta el baseline no supervisado y el protocolo de comparación supervisada."

display(conclusion_tecnica(
    respuesta=f"Isolation Forest (no supervisado): PR-AUC={pr_auc:.4f}, Recall={if_recall:.1%} @ Precision={if_prec:.1%} (umbral por costo C_FN=100). {xgb_txt}",
    siguiente_paso="Implementar pipeline de reentrenamiento con validación temporal y active learning: (1) scoring masivo no supervisado, (2) priorización de alertas para revisión humana, (3) reentrenamiento supervisado con etiquetas confirmadas, (4) monitoreo continuo de PR-AUC, tasa de alertas y drift. Métrica norte: sostener el trade-off recall/precision acordado con costos operativos.",
    reproducibilidad=f"Notebook + environment.yml + generador de datos en data/download_data.py. Split temporal día 1→día 2. Seeds fijadas (42). Dataset en data/raw/creditcard.csv (proxy sintético reproducible por defecto, reemplazable por ULB/Kaggle con mismo esquema)."
))
